# Final Project: Electra Fine Tune Model

---

**Student Name:** Mohammad Mahdi Mahboob

**Student ID:** 400387250 | mahbom2

**CodaBench Username:** mahbom2

---

### AI Tool Usage Declaration

AI was used to learn the interfaces necessary for training the model (e.g. `transformers`, `AutoTokenizer`,
`Datasets`), debug errors in code, and learn how to save and load models to prevent retraining. No code from this
notebook was uploaded to any AI, only chat interfaces were used.

| Model Name | Hardware Type | Time Used | Provider | Compute Region |
| :--- | :--- | :--- | :--- | :--- |
| Google Gemini 3 | TPUv5 Chip | 24 hrs | Google Cloud Platform | northamerica-northeast1 |

The hardware and provider information was reused from a previous homework assignment for this course.

The amount of CO2 emissions calculated is 0.2 kg, which have already been offset by the provider.

In [1]:
# Install required packages (uncomment as needed)
%pip install evaluate
# !pip install sentencepiece  # needed for some tokenizers

# Core imports
import random
import numpy as np
import pandas as pd

# HuggingFace
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, set_seed
import evaluate

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
set_seed(SEED)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00


---
# Electra Base

## 1.1 Dataset Loading & Exploration

In [2]:
def full_text(df: pd.DataFrame):
    df['text'] = '[SENDER] ' + df['sender'].fillna('') + ' [SUBJECT] ' + df['subject'].fillna('') + ' [BODY] ' + df['body'].fillna('')

In [30]:
# TODO: Load + preprocess dataset(s)
train_csv = '/content/data/train.csv'
val_csv = '/content/data/val.csv'
test_csv = '/content/data/test.csv'

train_df = pd.read_csv(train_csv)
val_df = pd.read_csv(val_csv)
test_df = pd.read_csv(test_csv)

full_text(train_df)
full_text(val_df)
full_text(test_df)

In [31]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)
test_ds = Dataset.from_pandas(test_df)

In [6]:
print(train_df.head)

<bound method NDFrame.head of                                                   sender  \
0           Leanna Hooks <kglgilqgyduehc@avnetgroup.com>   
1                                             al@mpsc.ph   
2                  wantEnjoy <hand@verticalcircuits.com>   
3                                             Lochner <>   
4      "Joe Marie J. Maja" <maja@robotics.is.tohoku.a...   
...                                                  ...   
10495           BREAKING NEWS <breakingnews@foxnews.com>   
10496      CNN Alerts <scheerkl_1986@interdesign.com.pl>   
10497           Leticia Herrera <cbkarnack@finalcom.net>   
10498                 Steve Holden <gsdsz@holdenweb.com>   
10499            "Kirk Umfleet" <Sidney22@webmail.co.za>   

                                                receiver  \
0                                      jreitme@enron.com   
1                                        <ilug@linux.ie>   
2                              theorize@plg.uwaterloo.ca   
3        

In [7]:
train_head = train_ds[0:10]['text']
print(train_head)

['[SENDER] Leanna Hooks <kglgilqgyduehc@avnetgroup.com> [SUBJECT] A new major market score each week [BODY] "Stock Watch A|ert" this morning are Wysak Petroleum (WYSK), Key\nEnergy Services, Inc. (Pink Sheets: KEGS), Medify So|utions (MFYS),\nSequoia Interests Corporation (SQNC).\n\nWysak Petro|eum (WYSK)\nCurrent Price:   .225\n\nWysak Petro|eum announces the signing of a Letter of Intent with the European\nCommission Ba|tic Renewab|e Energy Centre (EC BREC) to assist Wysak Petroleum in\nthe deve|opment of the Wysak Wind Power Project.\n\nEC BREC and Wysak have signed a LOI in respect to the development of a\nfu||-sized Commercial Wind Power Project in Europe. This |etter states that EC\nBREC can support Wysak in matters such as financial structuring and investment,\nregulatory issues, government policies, negotiations, wind techno|ogies, and\nother aspects re|ating to Wind Power.\n\nAbout the Wysak Wind Project\n\nThis deve|opment wil| be up to a maximum 90Mw in size and cost upwards

In [8]:
f1_metric = evaluate.load("f1")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [9]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return f1_metric.compute(predictions=predictions, references=labels)

---
## 1.2 Fine-tuning Pretrained Models (#3.1)

Select **two** different pretrained transformer models (e.g. BERT, RoBERTa, DistilBERT, DeBERTa, ModernBERT, ELECTRA, T5, …).  
The models must **not** already be fine-tuned on your specific task.

### Fine-tuned Model 1

In [10]:
MODEL_1_NAME = 'google/electra-base-discriminator'

---

#### New Model Setup

In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_1_NAME)
model = AutoModelForSequenceClassification.from_pretrained( MODEL_1_NAME, num_labels=2)

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

ElectraForSequenceClassification LOAD REPORT from: google/electra-base-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
electra.embeddings_project.bias                   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
electra.embeddings_project.weight                 | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.bias                             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- 

---

#### Load Saved Model

In [ ]:
from zipfile import ZipFile

zip_name = '/content/ft_electra.zip'
with ZipFile(zip_name, 'r') as zip_f:
  zip_f.extractall('electra-ftm')

In [ ]:
model_1_path = '/content/electra-ftm'

tokenizer = AutoTokenizer.from_pretrained(model_1_path)
model = AutoModelForSequenceClassification.from_pretrained(model_1_path)

---

#### Running

In [12]:
def tknz(data):
    return tokenizer(data['text'], padding='max_length', truncation=True)

In [32]:
tok_train = train_ds.map(tknz, batched=True)
tok_val = val_ds.map(tknz, batched=True)
tok_test = test_ds.map(tknz, batched=True)

Map:   0%|          | 0/10500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2250 [00:00<?, ? examples/s]

Map:   0%|          | 0/2250 [00:00<?, ? examples/s]

In [14]:
from transformers import DataCollatorWithPadding
data_col = DataCollatorWithPadding(tokenizer=tokenizer)

In [23]:
train_args = TrainingArguments(
    output_dir='/content/electra_ft',
    eval_strategy="steps",
    eval_steps=100,
    logging_steps=100,
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=8,
    eval_accumulation_steps=1,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    fp16=True,
    group_by_length=True,
    gradient_checkpointing=True,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy='steps',
    save_total_limit=1,
    load_best_model_at_end=True,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    data_collator=data_col,
    compute_metrics=compute_metrics
)

In [24]:
trainer.train()

Step,Training Loss,Validation Loss,F1
100,0.397323,0.072901,0.985261
200,0.301662,0.064328,0.987489


Step,Training Loss,Validation Loss,F1
100,0.397323,0.072901,0.985261
200,0.301662,0.064328,0.987489
300,0.272150,0.042291,0.989791
400,0.221265,0.077753,0.983813
500,0.112354,0.106672,0.981507
600,0.126901,0.055648,0.989324
700,0.077052,0.078563,0.986179
800,0.064388,0.052646,0.988403
900,0.013134,0.054117,0.990663


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=987, training_loss=0.1642861274479612, metrics={'train_runtime': 1288.1299, 'train_samples_per_second': 24.454, 'train_steps_per_second': 0.766, 'total_flos': 8287998243840000.0, 'train_loss': 0.1642861274479612, 'epoch': 3.0})

---

#### Save

In [ ]:
save_dir = '/content/model_ft_electra'
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

In [ ]:
import shutil
zip_name = 'ft_electra'
shutil.make_archive(zip_name, 'zip', save_dir)

In [ ]:
from google.colab import files
files.download(f'/content/{zip_name}.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

#### Test

In [38]:
trainer.args.group_by_length = False
raw_pred = trainer.predict(tok_test)
test_y_pred = np.argmax(raw_pred.predictions, axis=-1)

In [42]:
from google.colab import files
submission_df = pd.DataFrame(test_y_pred, columns=['label'])
sub_file = '/content/submission.csv'
print(submission_df.head())
submission_df.to_csv(sub_file, index=False)

files.download(f'{sub_file}')

   label
0      1
1      0
2      1
3      1
4      0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## Code Attribution

List any code snippets, tutorials, or resources you referenced below. Include URLs and a short description of what was adapted.

| Source | URL | What was adapted |
|--------|-----|------------------|
| HuggingFace Trainer tutorial | https://huggingface.co/docs/transformers/training | Fine-tuning loop scaffold |
| *(add more rows as needed)* | | |